# 01 – Primeros Pasos con `pyglenn`

**`pyglenn`** es un calculador de propiedades termoquímicas ligero y sin
dependencias externas. Reconstruye tres propiedades molares en el estado
estándar como funciones analíticas de la temperatura,

$$C_p^\circ(T), \qquad H^\circ(T), \qquad S^\circ(T),$$

a partir de **coeficientes polinomiales de la NASA** almacenados en una base de
datos **SQLite** empaquetada. La base de datos se distribuye con el paquete y
contiene aproximadamente **2030 especies químicas** (gases y fases condensadas)
distribuidas en **3772 intervalos de temperatura**, derivadas del conjunto de
datos `thermo.inp` del NASA Glenn.

Este primer cuaderno cubre los fundamentos:

1. Conexión a la base de datos empaquetada
2. Inspección del contenido de la base de datos
3. Búsqueda de especies
4. Cálculo de $C_p^\circ$, $H^\circ$ y $S^\circ$ a una temperatura dada
5. Comprensión de los valores devueltos
6. Diferencias de entalpía y tablas de propiedades
7. Manejo elegante de errores

> **Autor del paquete:** Dr. Reginaldo G. Leão Jr. — GESESC / IFMG.

## Contenido

1. [Conectando a la base de datos](#1-conectando-a-la-base-de-datos)
2. [Inspeccionando la base de datos](#2-inspeccionando-la-base-de-datos)
3. [Buscando especies](#3-buscando-especies)
4. [Calculando propiedades a una temperatura](#4-calculando-propiedades-a-una-temperatura)
5. [Diferencias de entalpía (calor sensible)](#5-diferencias-de-entalpía-calor-sensible)
6. [Tablas de propiedades para varias temperaturas](#6-tablas-de-propiedades-para-varias-temperaturas)
7. [Manejo de errores](#7-manejo-de-errores)

## Instalación

`pyglenn` solo requiere Python ≥ 3.9 (SQLite3 está en la biblioteca estándar).
Instálelo desde PyPI, conda-forge o desde el código fuente:

```bash
pip install pyglenn
# o vía conda:
conda install conda-forge::pyglenn
# o, desde un repositorio local:
pip install .
```

Los comandos siguientes asumen que el paquete ya se puede importar.

In [ ]:
from pyglenn import ThermochemicalCalculator, R

_INDEX = {}

def species_id(calc, name):
    """Devuelve el id en la base de datos de la especie cuyo *nombre* coincide exactamente.

    ``get_available_species`` busca subcadenas tanto en el nombre como en la
    fórmula y limita el resultado a 20 filas, por lo que nombres cortos como
    ``"O2"`` pueden quedar desplazados por entradas como ``"AL2O2"`` o
    ``"CO2"``. Para ser robustos, construimos un índice completo nombre -> id
    una sola vez (almacenado en caché durante la sesión) y buscamos el nombre
    exacto en él.
    """
    if not _INDEX:
        _INDEX.update({s["name"]: s["id"] for s in calc.get_available_species("")})
    if name not in _INDEX:
        raise ValueError(f"Especie {name!r} no encontrada en la base de datos")
    return _INDEX[name]

print("Constante universal de los gases R =", R, "J/(mol.K)")


## 1. Conectando a la base de datos

`ThermochemicalCalculator` es el punto de entrada de alto nivel. Como gestiona
una conexión SQLite, el patrón recomendado es el **administrador de contexto**,
que abre la conexión al entrar y la cierra al salir — incluso si se lanza una
excepción:

```python
with ThermochemicalCalculator() as calc:
    ...  # calc está conectado aquí
# conexión cerrada automáticamente aquí
```

No se necesita ninguna ruta: el constructor usa por defecto el archivo
`thermo.db` empaquetado dentro del paquete.

In [ ]:
with ThermochemicalCalculator() as calc:
    print("¿Conectado? ", calc.connected)
print("¿Conectado después del bloque?", calc.connected)

### Gestión manual de la conexión

Si prefiere un control explícito (por ejemplo, dentro de un servicio de larga
duración), puede llamar a `connect()` y `close()` manualmente. `connect()`
devuelve `True` en caso de éxito.

In [ ]:
calc = ThermochemicalCalculator()
ok = calc.connect()
print("connect() ->", ok)
print("calc.connected ->", calc.connected)
calc.close()
print("después de close ->", calc.connected)

## 2. Inspeccionando la base de datos

El objeto de consulta subyacente se expone como `calc.db` (una instancia de
`ThermoDBQuery`). Su método `get_statistics()` proporciona una visión general
rápida del conjunto de datos.

In [ ]:
with ThermochemicalCalculator() as calc:
    stats = calc.db.get_statistics()

for key, value in stats.items():
    print(f"{key:22s}: {value}")

Observe que `species_by_phase` distingue especies en estado **gaseoso** (gas) de
las especies **condensadas** (condensed — líquido/sólido). La base de datos
mezcla ambos, por lo que al buscar un combustible o un producto puede encontrar
varias entradas — una fase gaseosa, una fase líquida y, a veces, múltiples
formas cristalinas.

## 3. Buscando especies

`get_available_species(pattern)` devuelve una lista de diccionarios. Con un
`pattern` no vacío, realiza una coincidencia **parcial** tanto en el nombre como
en la fórmula química (limitado a 20 resultados); con un patrón vacío, pagina
por todo el catálogo.

In [ ]:
with ThermochemicalCalculator() as calc:
    matches = calc.get_available_species("H2O")

print(f"{len(matches)} coincidencias para 'H2O':\n")
for s in matches:
    print(f"  id={s['id']:5d}  {s['name']:18s}  {s['phase']:9s}  M={s['molecular_weight']:.4f} g/mol")

Como la búsqueda es por subcadena, la consulta anterior devuelve `CH2OH`,
`H2O2`, `NH2OH`, ... además del agua misma. Cuando necesite una especie
específica, filtre por el nombre **exacto**. Eso es exactamente lo que hace el
auxiliar `species_id` definido en el preámbulo:

In [ ]:
with ThermochemicalCalculator() as calc:
    for name in ["O2", "N2", "CO2", "H2O", "CH4", "Ar"]:
        sid = species_id(calc, name)
        print(f"{name:5s} -> id en base de datos {sid}")

### Navegando por todo el catálogo

Pasar una cadena vacía pagina por todas las ~2030 especies. Aquí solo las
contamos y previsualizamos las primeras.

In [ ]:
with ThermochemicalCalculator() as calc:
    everything = calc.get_available_species("")

print(f"Total de especies en el catálogo: {len(everything)}\n")
print("Primeras 8 (en orden alfabético):")
for s in everything[:8]:
    print(f"  {s['name']:20s}  {s['phase']}")

## 4. Calculando propiedades a una temperatura

`calculate_properties(species_id, temperature_K)` es el método central. Devuelve
un diccionario de propiedades en el estado estándar evaluadas a la temperatura
solicitada. Evaluemos el oxígeno molecular a 298,15 K y a 1000 K.

In [ ]:
with ThermochemicalCalculator() as calc:
    o2 = species_id(calc, "O2")
    for T in (298.15, 1000.0):
        props = calc.calculate_properties(o2, T)
        print(f"--- O2 a {T} K ---")
        for k, v in props.items():
            print(f"    {k:14s}: {v}")
        print()

### ¿Qué significan estos números?

| Clave | Significado | Unidades |
|-----|---------|-------|
| `temperature`   | La temperatura solicitada | K |
| `cp`            | Capacidad calorífica en el estado estándar $C_p^\circ(T)$ | J/(mol·K) |
| `s`             | Entropía absoluta en el estado estándar (Tercera Ley) $S^\circ(T)$ | J/(mol·K) |
| `h_relative`    | Entalpía molar estandarizada $H^\circ(T)$ en la escala NASA | J/mol |
| `temp_interval` | `[T_min, T_max]` del tramo polinomial utilizado | K |
| `species_name`  | Nombre de la especie resuelta | — |
| `phase`         | `gas` o `condensed` (condensado) | — |

Un punto crucial sobre **`h_relative`**: los polinomios de la NASA ajustan la
entalpía *estandarizada*, que **ya incluye la entalpía de formación**. Como
consecuencia:

* para una especie en su estado de referencia elemental (ej.: O₂, N₂, grafito),
  $H^\circ(298{,}15\,\text{K}) \approx 0$;
* para un compuesto, $H^\circ(298{,}15\,\text{K})$ es igual a su **entalpía de
  formación** $\Delta_f H^\circ$.

Es por eso que el `h_relative` del oxígeno arriba es esencialmente cero a
298,15 K. Esta propiedad se explota en cuadernos posteriores para calcular
entalpías de reacción directamente. Verificación rápida para el vapor de agua:

In [ ]:
with ThermochemicalCalculator() as calc:
    h2o = species_id(calc, "H2O")
    hf = calc.calculate_properties(h2o, 298.15)["h_relative"]
print(f"Entalpía estandarizada del H2O(g) a 298,15 K = {hf/1000:8.2f} kJ/mol")
print("Entalpía de formación de la literatura para H2O(g) = −241,83 kJ/mol")

## 5. Diferencias de entalpía (calor sensible)

El cambio de entalpía necesario para calentar un mol de una sustancia de $T_1$
a $T_2$ (su entalpía *sensible*) es $H^\circ(T_2) - H^\circ(T_1)$.
`calculate_enthalpy_change` lo calcula directamente.

In [ ]:
with ThermochemicalCalculator() as calc:
    n2 = species_id(calc, "N2")
    dH = calc.calculate_enthalpy_change(n2, 300.0, 1200.0)
print(f"Calentar 1 mol de N2 de 300 K a 1200 K requiere {dH/1000:.2f} kJ")

## 6. Tablas de propiedades para varias temperaturas

`get_properties_range(species_id, [T1, T2, ...])` evalúa las propiedades en una
lista de temperaturas y devuelve un diccionario indexado por temperatura. Las
temperaturas que estén fuera de todos los intervalos válidos se omiten
silenciosamente.

In [ ]:
temps = [300, 500, 800, 1200, 1800, 2500]
with ThermochemicalCalculator() as calc:
    co2 = species_id(calc, "CO2")
    table = calc.get_properties_range(co2, temps)

print(f"{'T [K]':>7} {'Cp [J/mol/K]':>14} {'S [J/mol/K]':>13} {'H [kJ/mol]':>12}")
for T in temps:
    p = table[T]
    print(f"{T:7.0f} {p['cp']:14.3f} {p['s']:13.3f} {p['h_relative']/1000:12.3f}")

Observe cómo $C_p$ y $S$ del CO₂ crecen fuertemente con la temperatura — una
marca distintiva de las moléculas poliatómicas cuyos modos vibracionales se
vuelven cada vez más excitados. El cuaderno 03 explora este comportamiento
gráficamente.

## 7. Manejo de errores

`pyglenn` es defensivo: en lugar de lanzar excepciones para fallos comunes de
consulta, devuelve `None` (y registra un mensaje). Dos casos comunes:

* la **temperatura solicitada está fuera de rango** para la especie;
* un método se llama **antes de conectar**.

El paquete también define una jerarquía de excepciones
(`ThermoCalcError` → `DatabaseNotConnectedError`, `SpeciesNotFoundError`,
`TemperatureOutOfRangeError`) que puede importar para sus propias capas de
validación.

In [ ]:
with ThermochemicalCalculator() as calc:
    o2 = species_id(calc, "O2")
    # O2 gas es válido hasta 20000 K, pero 50000 K está fuera de rango:
    out_of_range = calc.calculate_properties(o2, 50_000.0)
    print("calculate_properties a 50000 K ->", out_of_range)

# Llamar antes de conectar devuelve None en lugar de fallar:
lonely = ThermochemicalCalculator()
print("calculate_properties antes de connect() ->",
      lonely.calculate_properties(o2, 300.0))

In [ ]:
from pyglenn import (
    ThermoCalcError,
    DatabaseNotConnectedError,
    SpeciesNotFoundError,
    TemperatureOutOfRangeError,
)

# Todas heredan de ThermoCalcError, así que puede capturar toda la familia de una vez.
for exc in (DatabaseNotConnectedError, SpeciesNotFoundError, TemperatureOutOfRangeError):
    print(f"{exc.__name__:28s} ¿es un ThermoCalcError? {issubclass(exc, ThermoCalcError)}")

## Resumen

Ahora sabe cómo:

- conectarse a la base de datos empaquetada (preferiblemente mediante el
  administrador de contexto);
- inspeccionarla con `get_statistics()`;
- buscar especies con `get_available_species()` y resolver nombres exactos;
- calcular $C_p^\circ$, $S^\circ$ y $H^\circ$ con `calculate_properties()`;
- interpretar `h_relative` como la entalpía estandarizada (que contiene
  $\Delta_f H^\circ$);
- construir tablas con `get_properties_range()` y diferencias de entalpía con
  `calculate_enthalpy_change()`.

**A continuación:** el cuaderno 02 abre la caja negra de los polinomios de la
NASA y muestra cómo se calculan estos números a partir de los coeficientes
brutos.